# J25 Biji Hitam Pecah — root-cause audit

Validation-only comparison of D0DIRECT and AF2LUMSAFE at lambda=0. Separates raw localization, final selection, and wrong-class confusion. No training and no locked-test evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
BRANCH='codex/j25-black-broken-audit'
REPO=Path('/content/coffee-bean-detection'); WORK=Path('/content')
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','gdown'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab')
experiment_roots=[Path('/content/drive/MyDrive/Coffee_Bean_Detection/experiments'),*Path('/content/drive/.shortcut-targets-by-id').glob('*/Coffee_Bean_Detection/experiments')]
def find_root(name, result_name):
    matches=[root/name for root in experiment_roots if (root/name/'val_reports'/result_name).is_file()]
    if not matches: raise FileNotFoundError(f'{name}/{result_name} tidak ditemukan di Drive')
    return matches[0]
D0_ROOT=find_root('coffee-standard-j25-af2-direct-v2','D0DIRECT_seed42_result.json')
SAFE_ROOT=find_root('coffee-standard-j25-af2-luminance-safe-v1','AF2LUMSAFE_seed42_result.json')
PROJECT=D0_ROOT.parents[1]
D0_RESULT=D0_ROOT/'val_reports/D0DIRECT_seed42_result.json'
SAFE_RESULT=SAFE_ROOT/'val_reports/AF2LUMSAFE_seed42_result.json'
D0_CHECKPOINT=D0_ROOT/'D0DIRECT/D0DIRECT_seed42/weights/best.pt'
SAFE_CHECKPOINT=SAFE_ROOT/'AF2LUMSAFE/AF2LUMSAFE_seed42/weights/best.pt'
for checkpoint in (D0_CHECKPOINT,SAFE_CHECKPOINT):
    if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
print('D0:',D0_CHECKPOINT); print('SAFE:',SAFE_CHECKPOINT)


In [ ]:
from coffee_detector.analysis.coffee_standard_j25_thesis_provenance import audit_j25_thesis_provenance
from coffee_detector.data.prepare_coffee_standard_j25_source_split import prepare_j25_source_split
ARCHIVE=WORK/'data_aug_11.zip'
if not ARCHIVE.is_file(): subprocess.run([sys.executable,'-m','gdown','https://drive.google.com/uc?id=1AofT7VbiNFM8ul-0vyCAKj7Rp4j5OX0f','-O',str(ARCHIVE)],check=True)
PROVENANCE=WORK/'coffee_standard_j25_thesis_provenance.json'
provenance=audit_j25_thesis_provenance(ARCHIVE,PROVENANCE)
if not provenance['decision'].startswith('PASS'): raise RuntimeError(f'Provenance gagal: {provenance["decision"]}')
DATA=WORK/'coffee-standard-j25-train-siblings-v2'
if DATA.exists(): shutil.rmtree(DATA)
contract=prepare_j25_source_split(ARCHIVE,DATA,seed=42,retain_train_siblings=True)
CONTRACT=DATA/'coffee_standard_j25_train_siblings_summary.json'
OUT=PROJECT/'experiments/coffee-standard-j25-black-broken-audit-v1'; OUT.mkdir(parents=True,exist_ok=True)
SUMMARY=OUT/'black_broken_root_cause.json'
print('DATA:',contract['images'],'| OUT:',OUT)


In [ ]:
LOG=OUT/'black_broken_root_cause_run.log'
command=[sys.executable,'-u','-m','coffee_detector.analysis.coffee_standard_j25_black_broken_audit','--data-root',str(DATA),'--development-contract',str(CONTRACT),'--provenance-summary',str(PROVENANCE),'--d0-checkpoint',str(D0_CHECKPOINT),'--d0-result',str(D0_RESULT),'--safe-checkpoint',str(SAFE_CHECKPOINT),'--safe-result',str(SAFE_RESULT),'--output',str(SUMMARY),'--device','0','--authorize-diagnostic']
print('MENJALANKAN VALIDATION-ONLY ROOT-CAUSE AUDIT | log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
minutes=0
while process.poll() is None:
    time.sleep(120); minutes+=2; print(f'Masih berjalan: {minutes} menit',flush=True)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'Audit gagal: {process.returncode}')
result=json.loads(SUMMARY.read_text())
print('SUPPORT:',result['target_support'])
print('ATTRIBUTION:',result['attribution'])
print('TRAINING:',result['training_executed'],'| TEST:',result['test_opened'])


In [ ]:
import pandas as pd
rows=[]
for model,stages in result['models'].items():
    for stage,values in stages.items():
        rows.append({'model':model,'stage':stage,**{key:values[key] for key in ('targets','accessible','matched','correct_class','wrong_class','proposal_accessibility','matched_recall','localization_conditioned_class_accuracy')},'wrong_destinations':values['wrong_destinations']})
table=pd.DataFrame(rows)
display(table)
print('ATTRIBUTION:',result['attribution'])
print('SUMMARY:',SUMMARY)
